In [ ]:
# https://medium.com/@rehan020345/understanding-temporal-convolutional-networks-tcns-from-cnn-basics-to-full-sequence-mastery-57d0804ad8c8
# https://www.geeksforgeeks.org/deep-learning/model-fit-in-tensorflow/
# https://arxiv.org/pdf/2407.16387
# https://dl.acm.org/doi/pdf/10.1145/3534594
# https://medium.com/@amit25173/temporal-convolutional-network-an-overview-4d2b6f03d6f8
# https://unit8.com/resources/temporal-convolutional-networks-and-forecasting/
# https://www.kaggle.com/code/christofhenkel/temporal-convolutional-network
# https://unit8co.github.io/darts/examples/05-TCN-examples.html

# Recurrent Neural Nets: https://towardsdatascience.com/temporal-loops-intro-to-recurrent-neural-networks-for-time-series-forecasting-in-python-b0398963dc1f/
# https://www.kaggle.com/code/carlmcbrideellis/temporal-convolutional-network-using-keras-tcn

# https://pub.aimind.so/temporal-convolutional-networks-tcn-and-dilated-tcn-for-time-series-forecasting-921e361d316e

# Note:
- GenAI (specifically Claude-Sonnet-4.5) was used for research in this exploratory work on TOM. This was in accordance with level 2X on brightspace. Specific usage will be highlighted through commented code.

- Unfortunately, this work was a zero sum. My teammate Casey had a much better attempt with the algorithm she found. This work on TOM was exploratory, and did not produce the desired results.

# Imports

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

import joblib

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Model

In [ ]:
# models inputs
inputs = keras.Input(shape = (200, 11))
x = inputs

In [ ]:
# dilated TCN with
for i, filters in enumerate([16, 32, 32, 64]):
  # Below from: https://medium.com/@rehan020345/understanding-temporal-convolutional-networks-tcns-from-cnn-basics-to-full-sequence-mastery-57d0804ad8c8
  # dilation: solves the issue of expanding the models memory without making it massively deep
  # this is done by spacing out the kernels look back points
    dilation = 2 ** i # dilation increases exponentially

    # level 1
    conv1 = keras.layers.Conv1D(filters, 3, dilation_rate=dilation, padding='causal')(x) # standard 1D conv layer for feature extraction
    conv1 = keras.layers.BatchNormalization()(conv1) # batch normalisation to bring all inpuits into similar range
    conv1 = keras.layers.Activation('relu')(conv1) # activation layer ReLU - same as used for ARNIE
    conv1 = keras.layers.Dropout(0.1)(conv1) # Dropout to prevent overfitting

    # level 2
    conv2 = keras.layers.Conv1D(filters, 3, dilation_rate=dilation, padding='causal')(conv1)
    conv2 = keras.layers.BatchNormalization()(conv2)
    conv2 = keras.layers.Activation('relu')(conv2)
    conv2 = keras.layers.Dropout(0.1)(conv2)
    x = keras.layers.Add()([x, conv2])
    x = keras.layers.Activation('relu')(x)

# GenAI was used here for help with the attention pooling.
# This paper: https://www.nature.com/articles/s41598-024-57912-3 suggested it as a way of improving the models ability to distinguish
# different activity types (good for different flight pattenrs), but I did not understand how to implement it. GenAI provided guidance on the below implementation.

# attention pooling across times
attention_scores = keras.layers.Dense(1)(x)
attention_weights = keras.layers.Softmax(axis=1)(attention_scores)
x = keras.layers.Multiply()([x, attention_weights])
x = keras.layers.Lambda(lambda x: tf.reduce_sum(x, axis=1))(x)

# regression
# Dense layer  pools everything together for models final output
x = keras.layers.Dense(32, activation='relu')(x)
outputs = keras.layers.Dense(15)(x) # 15 outputs (arbitrary choice)

model = keras.Model(inputs, outputs)

# compile the model
# Learning rate: extremely important for ARNIE, never experimented with here because the model never came close to working
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0),
    loss='mse',
    metrics=['mae']
)

# standard loss function used ('mse' - mean square error)

# Data generation

# Note:
- The flight data generated below uses a LOT of arbitrary values for everything from spiral radii to temperature growth factors. The idea was to simulate 'realistic' flight data as best as possible for as many different flight patterns as possible, and train the model. If that model was able to function on that data, this data would then be combined with real flight data from other balloon missions (NOAA seemed like the best option). However, I could not make the model work on this simulated data, and TOM is now dead.

In [ ]:
# make simulated flight data for different flying patterns
# https://www.geeksforgeeks.org/numpy/numpy-cumsum-in-python/
# np.cumsum used extensivley here - simulates flight conditions by doing a cumulative sum of the elements in an array
# Example, horiz_drift = radius * np.cos(angle) + np.cumsum(np.random.randn(n_steps) * 0.1) * dt
# This adds random noise drift based on the step size - drift grows, like in dead reckoning.

def generate_flight_data(duration, dt, pattern):
  n_steps = int(duration/dt)
  time = np.linspace(0, duration, n_steps)

  # pattern 1 - going up in a spiral pattern
  if pattern == 'ascending_spiral':
    radius = 10
    altitude = 5 * time
    angle = time * 0.5
    horiz_drift = radius * np.cos(angle) + np.cumsum(np.random.randn(n_steps) * 0.1) * dt
    ydrift = radius * np.sin(angle) + np.cumsum(np.random.randn(n_steps) * 0.1) * dt

  # directional changes at over time at points
  elif pattern == 'waypoints':
    segment_duration = duration / 4 # 4 segments, 1 for each cardinal direction
    altitude = 50 + 2 * time
    horiz_drift = np.zeros(n_steps)
    ydrift = np.zeros(n_steps)

    for i in range(n_steps):
            t_seg = (time[i] % segment_duration) / segment_duration
            direction = int(time[i] / segment_duration) % 4

            if direction == 0:                  # N
              ydrift[i] = 20 * t_seg
            elif direction == 1:                # E
              horiz_drift[i] = 20
              ydrift[i] = 20 + 20 * t_seg
            elif direction == 2:                # S
              horiz_drift[i] = 20 + 20 * t_seg
              ydrift[i] = 40
            else:                               # W
              horiz_drift[i] = 40
              ydrift[i] = 40 - 20 * t_seg

  # going all over the place - high acceleration
  elif pattern == 'aggressive':
    altitude = 80 + 30 * np.sin(time * 0.3) + np.cumsum(np.random.randn(n_steps) * 0.5) * dt
    horiz_drift = 25 * np.sin(time * 0.5) + np.cumsum(np.random.randn(n_steps) * 2) * dt
    ydrift = 25 * np.cos(time * 0.5) + np.cumsum(np.random.randn(n_steps) * 2) * dt

  # up in a straight line
  elif pattern == 'straight_climb':
    altitude = 50 + 10 * time
    horiz_drift = 5 * time + np.cumsum(np.random.randn(n_steps) * 0.05)
    ydrift = np.cumsum(np.random.randn(n_steps) * 0.05) * dt  # very small drift

  # constant velocity
  elif pattern == 'const_vel':
    altitude = 100 + np.cumsum(np.random.randn(n_steps) * 0.1) * dt # altitude stays near constant
    horiz_drift = 15 * time + np.cumsum(np.random.randn(n_steps) * 0.1) * dt
    ydrift = 8 * time + np.cumsum(np.random.randn(n_steps) * 0.1) * dt

  # free fall (i.e., after balloon burst)
  elif pattern == 'free_fall':
    altitude = 50 + 10 * time - 0.5 * 9.81 * time**2
    horiz_drift = np.zeros(n_steps)
    ydrift = np.zeros(n_steps)

  else: # this is the random pattern i was using orignally before adding flight patterns
    altitude = 5* time + np.cumsum(np.random.randn(n_steps) * 0.3) * dt
    wind_speed = 5 + 10 * np.sin(altitude / 5000) # altitude dependent wind - higher alt gives
    horiz_drift = np.cumsum(wind_speed * np.cos(time/100)) * dt
    ydrift = np.cumsum(np.random.randn(n_steps) * 1.5) * dt

  positions = np.column_stack([horiz_drift, ydrift, altitude])

  # get the difference between drift positions to get drfit velocity
  # https://www.geeksforgeeks.org/python/numpy-diff-in-python/
  horiz_velocity = np.diff(horiz_drift)
  y_velocity = np.diff(ydrift)

  # heading
  heading = np.arctan2(y_velocity, horiz_velocity)

  # https://www.geeksforgeeks.org/python/numpy-pad-function-in-python/
  # pad to match length of array
  heading = np.pad(heading, (0, 1), mode='edge')

  # https://numpy.org/doc/2.1/reference/generated/numpy.convolve.html
  heading_smooth = np.convolve(heading, np.ones(50)/50, mode='same')

  # simulate mabnetometer
  mag_horiz = 25 # units are micro teslas
  mag_vert = 43

  mag_x = mag_horiz * np.cos(heading_smooth)
  mag_y = mag_horiz * np.sin(heading_smooth)
  mag_z = mag_vert
  mag_noise  = np.random.randn(n_steps, 3) * 1.5 # add random noise
  mag_x *= 1.02 # add more random noise
  mag_y *= 0.98

  mag = np.column_stack([mag_x, mag_y, mag_z])
  mag += mag_noise

  # using calculus to get velocity and accel from pos.
  velocity = np.diff(positions, axis = 0) * 50
  accel = np.diff(positions, axis=0, n=2) * 50**2
  accel = np.pad(accel, ((0, 2), (0, 0)), mode='edge') # must be padded to match lengths

  # add IMU
  accelbias = np.array([0.02, -0.015, 0.03]) # constant bias
  accelnoise = np.random.randn(*accel.shape) * 0.2 # random noise
  accel = accel +accelbias + accelnoise

  # simulation of gyro motion
  gyro = np.column_stack([
    0.5 * np.sin(time/30), #x
    0.5 * np.cos(time/30), #y
    0.2 * np.sin(time/100) #z
  ])

  gyro += np.random.randn(n_steps, 3) * 0.03 # add noise
  gyro += np.array([0.01, -0.008, 0.012]) # add bias

  # simulate barometer data
  pressure = 101325 * np.exp(-altitude / 8500)
  pressure += np.random.randn(n_steps) * 0.3 # add noise

  # use a simple atmospheric temperature model
  temperature = 15 - altitude * 0.006
  temp += np.random.randn(n_steps) * 0.3

  X_init = np.column_stack([
    accel[:, 0], accel[:, 1], accel[:, 2],  # ax, ay, az
    gyro[:, 0], gyro[:, 1], gyro[:, 2],     # gx, gy, gz
    mag[:, 0], mag[:, 1], mag[:, 2],        # mx, my, mz
    pressure,                                # pressure
    temperature                              # temperature
  ])

  return X_init, positions

In [ ]:
# generate flight data
patterns = ['ascending_spiral', 'waypoints', 'aggressive', 'random', 'straight_climb', 'const_vel', 'free_fall']

# empty lists for flight data
train_X_list = []
train_pos_list = []
test_X_list = []
test_pos_list = []

# perform splitting of data
for i, pattern in enumerate(patterns):
    print(f"Flight {i+1}/{len(patterns)}: {pattern}")
                                # dur, dt, pattern
    X, pos = generate_flight_data(600, 0.02, pattern)

    split_idx = int(0.8 * len(X))
    train_X_list.append(X[:split_idx])
    train_pos_list.append(pos[:split_idx])
    test_X_list.append(X[split_idx:])
    test_pos_list.append(pos[split_idx:])

# concatenate into single arrays
X_train_raw = np.concatenate(train_X_list, axis=0)
pos_train_raw = np.concatenate(train_pos_list, axis=0)
X_test_raw = np.concatenate(test_X_list, axis=0)
pos_test_raw = np.concatenate(test_pos_list, axis=0)

# Note:
- Below, GenAI was used. Specifically, it assisted in the development of the the windowing function, and the evaluation code shown below.

- While I found some resources on the windowing function:
https://kr.mathworks.com/help/predmaint/ref/tcndetector.html, https://unit8.com/resources/temporal-convolutional-networks-and-forecasting/

I did not understand how to implement them. Given that this was exploratory work to test TOMs feasability, GenAI was used to help below.

# MAJOR ERROR MADE HERE - THAT EXPLAINS WHY THE MODEL PERFORMED SO WELL:
- scaler_pos = StandardScaler()
- all_pos_train = np.concatenate(train_pos_list, axis=0)
- scaler_pos.fit(all_pos_train)

This is major data leakage, as suspected. Only the training data should be normalised using the scaler. The scaler learned statistics from validation/test data -> The model thus sees validation/test distribution during training. This led to performance metrics appearing to be way too good

In [ ]:
######## MAJOR ERROR ###########
scaler_pos = StandardScaler() # normalise data, crucial for this task
all_pos_train = np.concatenate(train_pos_list, axis=0) # get all data into one array
scaler_pos.fit(all_pos_train) # fit to training data - BIG MISTAKE - Model has now indirectly 'seen' ALL data
################################

# Developed with the assistance of GenAI. Please see above.
def create_windows(X, positions, window_size, predict_steps, stride=10, noise_std = 0.3):
  xwindows = []
  ywindows = []

  # sliding window for time series data
  for i in range(0, len(X) - window_size - predict_steps, stride):
    window = X[i : i+window_size] # inpuit window
    last_input_pos = positions[i + window_size - 1]

    # calculate displacement for each time step
    next_disp = []
    for j in range(1, predict_steps+ 1):
      future_pos = positions[i+window_size -1 + j]

      # add noise to 30% to make model robust
      use_noise = np.random.rand() < 0.3

      if use_noise:
                # simulate error in GPS
                position_noise = np.random.randn(3) * noise_std * np.sqrt(j)
                noisy_reference = last_input_pos + position_noise
                cumul_disp = future_pos - noisy_reference
      else:
                # most cases do not have simulated error
                cumul_disp = future_pos - last_input_pos

      next_disp.append(cumul_disp)

    xwindows.append(window)
    ywindows.append(np.array(next_disp).flatten()) # flatten prediction

  x_train  = np.array(xwindows)
  y_train  = np.array(ywindows)

  return x_train, y_train

In [ ]:
X_train_windows = []
y_train_windows = []
X_test_windows = []
y_test_windows = []

# generate windows for the TCN
# Apply scaler fitted to training data
for X_norm_segment, pos_segment in zip(
    [scaler_X.transform(x) for x in train_X_list],
    train_pos_list):

    # 200 timestep input, predcit 5 steps ahead
    X_w, y_w = create_windows(X_norm_segment, pos_segment, 200, 5, stride=10)
    X_train_windows.append(X_w)
    y_train_windows.append(y_w)

# Apply scaler fitted to test data (using same scaler)
for X_norm_segment, pos_segment in zip(
    [scaler_X.transform(x) for x in test_X_list],
    test_pos_list
):
    X_w, y_w = create_windows(X_norm_segment, pos_segment, 200, 5, stride=10)
    X_test_windows.append(X_w)
    y_test_windows.append(y_w)

In [ ]:
# concat all data into managable arrays
X_train = np.concatenate(X_train_windows, axis=0)
y_train = np.concatenate(y_train_windows, axis=0)
X_test = np.concatenate(X_test_windows, axis=0)
y_test = np.concatenate(y_test_windows, axis=0)

print(f"Train windows: {len(X_train)}, Test widnows: {len(X_test)}")
print(f"Train flights in data: {len(patterns)}")
print(f"Test has samples from {len(patterns)} flight types")

Train windows: 16660, Test windows: 4060
Train flights in data: 7
Test has samples from 7 flight types


# Training

In [ ]:
# adaptive learning rate to try and get the model to improve over time
reduce_lr = ReduceLROnPlateau(
    monitor = 'val_loss',
    factor = 0.5,
    patience = 10,
    min_lr = 1e-7,
    verbose = 1,
    cooldown=3
)

In [ ]:
model.fit(X_train, y_train, epochs = 200, verbose = 1, validation_data = (X_test, y_test), callbacks = [reduce_lr], batch_size = 128)

# the normalising scaler has to be saved so the data can be reverted
joblib.dump(scaler_X, 'scaler_X.pkl')

scaler_X = joblib.load('scaler_X.pkl')

Epoch 1/200
131/131 ━━━━━━━━━━━━━━━━━━━━ 38s 127ms/step - loss: 702.1648 - mae: 5.7760 - val_loss: inf - val_mae: 10610182186103118693088217792512.0000 - learning_rate: 0.0010
Epoch 2/200
131/131 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 67.1568 - mae: 1.9558 - val_loss: inf - val_mae: 3642193680895613275354381156352.0000 - learning_rate: 0.0010
Epoch 3/200
131/131 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 44.7533 - mae: 1.4707 - val_loss: inf - val_mae: 2003013731047504258457273368576.0000 - learning_rate: 0.0010
Epoch 4/200
131/131 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 49.1591 - mae: 1.4737 - val_loss: inf - val_mae: 934891426085527217814802268160.0000 - learning_rate: 0.0010
Epoch 5/200
131/131 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 23.0353 - mae: 1.1177 - val_loss: inf - val_mae: 2205775827323362026439521075200.0000 - learning_rate: 0.0010
Epoch 6/200
131/131 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 17.9349 - mae: 1.0002 - val_loss: inf - val_mae: 16850314838715428649207762

KeyboardInterrupt: 

In [ ]:
test_loss = model.evaluate(X_test, y_test)
y_pred_real = model.predict(X_test)

y_pred_reshaped = y_pred_real.reshape(-1, 5, 3)
y_test_reshaped = y_test.reshape(-1, 5, 3)

errors = np.linalg.norm(y_pred_reshaped[:, 0, :] - y_test_reshaped[:, 0, :], axis=1)
print(f"Mean distance error: {np.mean(errors):.2f} meters")
print(f"Max distance error: {np.max(errors):.2f} meters")

axis_errors = np.abs(y_pred_reshaped[:, 0, :] - y_test_reshaped[:, 0, :])
print(f"\nPer-axis mean errors (x, y, z):")
print(f"X: {axis_errors[:, 0].mean():.4f}m")
print(f"Y: {axis_errors[:, 1].mean():.2f}m")
print(f"Z: {axis_errors[:, 2].mean():.2f}m")

In [ ]:
# patterns = ['ascending_spiral', 'waypoints', 'aggressive', 'random', 'straight_climb', 'const_vel', 'free_fall']

In [ ]:
test_flight_idx = 0  # (0-6)#

# test data for different flight patterns
X_test_consecutive = test_X_list[test_flight_idx]
pos_test_consecutive = test_pos_list[test_flight_idx]

# normalise using train fitted scalar
X_test_consecutive_norm = scaler_X.transform(X_test_consecutive)

X_test_seq, y_test_seq = create_windows(
    X_test_consecutive_norm,
    pos_test_consecutive,
    window_size=200,
    predict_steps=5,
    stride=10
)

print(f"Evaluating on flight pattern: {patterns[test_flight_idx]}")
print(f"Test sequence has {len(X_test_seq)} windows")

N = min(100, len(X_test_seq)) # limit the eval to the first 100 windows

start_pos = pos_test_consecutive[200] # start pos

# model makes predictions
y_pred_seq = model.predict(X_test_seq, batch_size = 128)

# GenAI was used here. The specific prompt given was 'I want to see how this model performs visually, I'm not sure how, can you show me?'
# I understood the model output displacements, but I didnt know how best to convert that into
# something resembling visual model predictions for dead reckoning

# GenAI then suggested the below implementation for cumulating the displacement and making useful plots.

# reshape to make into 3D arrays
y_pred_reshaped = y_pred_seq.reshape(-1, 5, 3)
y_test_reshaped = y_test_seq.reshape(-1, 5, 3)

# get reference positions for each window
window_starts = np.arange(0, len(X_test_consecutive_norm) - 200 - 5, 10)[:N]

for ws in window_starts:
  reference_positions = np.array([pos_test_consecutive[ws]])

cumulative_pred = [reference_positions[0]]
cumulative_true = [reference_positions[0]]

# accumukate the displacements to simulate the trajectory
# like getting a map and marking points along the route
for i in range(len(y_pred_real)):
    for j in range(5): # for each of the 5 predicted steps
        cumulative_pred.append(cumulative_pred[-1] + y_pred_real[i, j])
        cumulative_true.append(cumulative_true[-1] + y_test_real[i, j])

# accumulate displacements to get the position
cumulative_pred = np.array(cumulative_pred)
cumulative_true = np.array(cumulative_true)

# get overall distance error
distance_err = np.linalg.norm(cumulative_pred - cumulative_true, axis=1)

print(f"Final drift error: {distance_err[-1]:.2f} m")
print(f"Mean drift error: {distance_err.mean():.2f} m")

# Plotting
fig = plt.figure(figsize=(24, 5))

ax1 = fig.add_subplot(131, projection='3d')
ax1.plot(cumulative_true[:, 0], cumulative_true[:, 1], cumulative_true[:, 2],
         'b-', label='True', linewidth=2)
ax1.plot(cumulative_pred[:, 0], cumulative_pred[:, 1], cumulative_pred[:, 2],
         'r--', label='Predicted', linewidth=2)
ax1.scatter(start_pos[0], start_pos[1], start_pos[2], c='green', s=100, label='Start')
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_zlabel('Z (m)')
ax1.set_title(f'3D Trajectory: {patterns[test_flight_idx]}')
ax1.legend()

ax2 = fig.add_subplot(132)

ax2.plot(cumulative_true[:, 2], label='True altitude', linewidth=2)
ax2.plot(cumulative_pred[:, 2], label='Predicted altitude', linewidth=2, linestyle='--')
ax2.set_ylabel('Altitude (m)')
ax2.set_title('Altitude Profile')

ax2.set_xlabel('Window #')
ax2.grid(True)

ax3 = fig.add_subplot(133)
ax3.plot(distance_err, linewidth=2)
ax3.set_xlabel('Window #')
ax3.set_ylabel('Position Error (m)')
ax3.set_title(f'Cumulative Drift (Final: {distance_err[-1]:.1f}m)')
ax3.grid(True)

plt.tight_layout()
plt.show()

print(f"Reference positions Z range: {reference_positions[:, 2].min():.1f} to {reference_positions[:, 2].max():.1f}")
print(f"Predicted deltas Z range: {y_pred_real[:N, 2].min():.1f} to {y_pred_real[:N, 2].max():.1f}")
print(f"True deltas Z range: {y_test_real[:N, 2].min():.1f} to {y_test_real[:N, 2].max():.1f}")

# Note:
This work on TOM did NOT produce a working model, or anything close. Particularly given the discovery of the data leakage while cleaning up this code, this model was far from being operational, as now even the promising* early results turn out to be incorrect.

*It was suspected from the outset that the results of this model were too good to be true. Teams such as the TinyODOM team managed to get a TCN model to work on edge hardware for a similar purpose (dead reckoning on edge hardware), but their work was significantly more sophisticated. In short, it CAN be done, but not by us. This was found to be true due to a rookie mistake made when making the normalising scaler for the data. While this is an essential step, it should be applied only to the training data. I applied it to all data, allowing the model to indirectly see all the data.

A lot was still learned in the exploration of TOM, especially regarding TCNs and dead reckoning.